# Experiment 1: Single Layer Perceptron with Activation Functions

**Objective**: Implement a Single Layer Perceptron and apply different activation functions (Sigmoid, ReLU, Tanh) on the output layer.

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 2. Define Activation Functions

In [ ]:
class ActivationFunctions:
    """Collection of activation functions and their derivatives."""
    
    @staticmethod
    def sigmoid(x):
        """Sigmoid activation: f(x) = 1 / (1 + e^-x)"""
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    @staticmethod
    def sigmoid_derivative(x):
        """Derivative of sigmoid: f'(x) = f(x) * (1 - f(x))"""
        s = ActivationFunctions.sigmoid(x)
        return s * (1 - s)
    
    @staticmethod
    def relu(x):
        """ReLU activation: f(x) = max(0, x)"""
        return np.maximum(0, x)
    
    @staticmethod
    def relu_derivative(x):
        """Derivative of ReLU: f'(x) = 1 if x > 0, else 0"""
        return (x > 0).astype(float)
    
    @staticmethod
    def tanh(x):
        """Tanh activation: f(x) = (e^x - e^-x) / (e^x + e^-x)"""
        return np.tanh(x)
    
    @staticmethod
    def tanh_derivative(x):
        """Derivative of tanh: f'(x) = 1 - tanh(x)^2"""
        return 1 - np.tanh(x) ** 2

## 3. Visualize Activation Functions

In [ ]:
# Create input range
x = np.linspace(-5, 5, 100)

# Plot activation functions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sigmoid
axes[0].plot(x, ActivationFunctions.sigmoid(x), 'b-', linewidth=2, label='Sigmoid')
axes[0].plot(x, ActivationFunctions.sigmoid_derivative(x), 'r--', linewidth=2, label='Derivative')
axes[0].set_title('Sigmoid Activation', fontsize=12, fontweight='bold')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)

# ReLU
axes[1].plot(x, ActivationFunctions.relu(x), 'b-', linewidth=2, label='ReLU')
axes[1].plot(x, ActivationFunctions.relu_derivative(x), 'r--', linewidth=2, label='Derivative')
axes[1].set_title('ReLU Activation', fontsize=12, fontweight='bold')
axes[1].set_xlabel('x')
axes[1].set_ylabel('f(x)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].axvline(x=0, color='k', linewidth=0.5)

# Tanh
axes[2].plot(x, ActivationFunctions.tanh(x), 'b-', linewidth=2, label='Tanh')
axes[2].plot(x, ActivationFunctions.tanh_derivative(x), 'r--', linewidth=2, label='Derivative')
axes[2].set_title('Tanh Activation', fontsize=12, fontweight='bold')
axes[2].set_xlabel('x')
axes[2].set_ylabel('f(x)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=0, color='k', linewidth=0.5)
axes[2].axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.savefig('../data/activation_functions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Single Layer Perceptron Implementation

In [ ]:
class SingleLayerPerceptron:
    """
    Single Layer Perceptron implementation with configurable activation function.
    
    Parameters:
    -----------
    n_features : int
        Number of input features
    activation : str
        Activation function: 'sigmoid', 'relu', or 'tanh'
    learning_rate : float
        Learning rate for gradient descent
    n_epochs : int
        Number of training epochs
    """
    
    def __init__(self, n_features, activation='sigmoid', learning_rate=0.01, n_epochs=100):
        self.n_features = n_features
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.activation_name = activation
        
        # Initialize weights and bias
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0
        
        # Set activation function and its derivative
        self._set_activation(activation)
        
        # Track training history
        self.loss_history = []
    
    def _set_activation(self, activation):
        """Set the activation function based on name."""
        activation_map = {
            'sigmoid': (ActivationFunctions.sigmoid, ActivationFunctions.sigmoid_derivative),
            'relu': (ActivationFunctions.relu, ActivationFunctions.relu_derivative),
            'tanh': (ActivationFunctions.tanh, ActivationFunctions.tanh_derivative)
        }
        if activation not in activation_map:
            raise ValueError(f"Unknown activation: {activation}. Choose from {list(activation_map.keys())}")
        self.activation, self.activation_derivative = activation_map[activation]
    
    def forward(self, X):
        """Forward pass: compute weighted sum and apply activation."""
        self.z = np.dot(X, self.weights) + self.bias
        self.output = self.activation(self.z)
        return self.output
    
    def compute_loss(self, y_true, y_pred):
        """Compute Mean Squared Error loss."""
        return np.mean((y_true - y_pred) ** 2)
    
    def backward(self, X, y_true, y_pred):
        """Backward pass: compute gradients and update weights."""
        n_samples = X.shape[0]
        
        # Compute gradient
        error = y_pred - y_true
        d_activation = self.activation_derivative(self.z)
        
        # Gradient for weights and bias
        dw = (1/n_samples) * np.dot(X.T, error * d_activation)
        db = (1/n_samples) * np.sum(error * d_activation)
        
        # Update weights and bias
        self.weights -= self.learning_rate * dw
        self.bias -= self.learning_rate * db
    
    def fit(self, X, y, verbose=True):
        """Train the perceptron."""
        self.loss_history = []
        
        for epoch in range(self.n_epochs):
            # Forward pass
            y_pred = self.forward(X)
            
            # Compute loss
            loss = self.compute_loss(y, y_pred)
            self.loss_history.append(loss)
            
            # Backward pass
            self.backward(X, y, y_pred)
            
            if verbose and (epoch + 1) % 20 == 0:
                print(f"Epoch {epoch+1}/{self.n_epochs}, Loss: {loss:.6f}")
        
        return self
    
    def predict(self, X):
        """Make predictions."""
        return self.forward(X)
    
    def predict_class(self, X, threshold=0.5):
        """Make binary class predictions."""
        return (self.predict(X) >= threshold).astype(int)

## 5. Generate and Preprocess Dataset

In [ ]:
# Generate synthetic binary classification dataset
X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
# Preprocessing: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Preprocessing: Feature scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled features - Mean: {X_train_scaled.mean(axis=0)}, Std: {X_train_scaled.std(axis=0)}")

In [ ]:
# Visualize the dataset
plt.figure(figsize=(8, 6))
plt.scatter(X_train_scaled[y_train == 0, 0], X_train_scaled[y_train == 0, 1], 
            c='blue', label='Class 0', alpha=0.6, edgecolors='k')
plt.scatter(X_train_scaled[y_train == 1, 0], X_train_scaled[y_train == 1, 1], 
            c='red', label='Class 1', alpha=0.6, edgecolors='k')
plt.xlabel('Feature 1 (scaled)')
plt.ylabel('Feature 2 (scaled)')
plt.title('Training Data Distribution', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Train Perceptrons with Different Activation Functions

In [ ]:
# Define activation functions to test
activations = ['sigmoid', 'relu', 'tanh']
models = {}
results = {}

for activation in activations:
    print(f"\n{'='*50}")
    print(f"Training Perceptron with {activation.upper()} activation")
    print('='*50)
    
    # Create and train model
    model = SingleLayerPerceptron(
        n_features=2,
        activation=activation,
        learning_rate=0.1,
        n_epochs=100
    )
    model.fit(X_train_scaled, y_train, verbose=True)
    
    # Store model
    models[activation] = model
    
    # Evaluate
    y_pred_train = model.predict_class(X_train_scaled)
    y_pred_test = model.predict_class(X_test_scaled)
    
    train_acc = np.mean(y_pred_train == y_train) * 100
    test_acc = np.mean(y_pred_test == y_test) * 100
    
    results[activation] = {'train_acc': train_acc, 'test_acc': test_acc}
    
    print(f"\nResults for {activation.upper()}:")
    print(f"  Training Accuracy: {train_acc:.2f}%")
    print(f"  Test Accuracy: {test_acc:.2f}%")

## 7. Compare Results

In [ ]:
# Plot training loss comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
colors = {'sigmoid': 'blue', 'relu': 'green', 'tanh': 'orange'}
for activation, model in models.items():
    axes[0].plot(model.loss_history, label=activation.capitalize(), 
                 color=colors[activation], linewidth=2)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Training Loss Comparison', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
x_pos = np.arange(len(activations))
train_accs = [results[a]['train_acc'] for a in activations]
test_accs = [results[a]['test_acc'] for a in activations]

width = 0.35
axes[1].bar(x_pos - width/2, train_accs, width, label='Train', color='steelblue')
axes[1].bar(x_pos + width/2, test_accs, width, label='Test', color='coral')

axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Comparison by Activation Function', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([a.capitalize() for a in activations])
axes[1].legend()
axes[1].set_ylim([0, 100])
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (train, test) in enumerate(zip(train_accs, test_accs)):
    axes[1].annotate(f'{train:.1f}%', xy=(i - width/2, train + 2), ha='center', fontsize=9)
    axes[1].annotate(f'{test:.1f}%', xy=(i + width/2, test + 2), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/perceptron_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Visualize Decision Boundaries

In [ ]:
def plot_decision_boundary(model, X, y, title):
    """Plot the decision boundary for a trained model."""
    # Create mesh grid
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    plt.scatter(X[y == 0, 0], X[y == 0, 1], c='blue', label='Class 0', 
                edgecolors='k', alpha=0.6)
    plt.scatter(X[y == 1, 0], X[y == 1, 1], c='red', label='Class 1', 
                edgecolors='k', alpha=0.6)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(title, fontsize=11, fontweight='bold')
    plt.legend(loc='upper right')

In [ ]:
# Plot decision boundaries for all activation functions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (activation, model) in enumerate(models.items()):
    plt.sca(axes[idx])
    test_acc = results[activation]['test_acc']
    plot_decision_boundary(
        model, 
        X_test_scaled, 
        y_test, 
        f'{activation.capitalize()} (Acc: {test_acc:.1f}%)'
    )

plt.tight_layout()
plt.savefig('../data/decision_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 1 SUMMARY: Single Layer Perceptron")
print("="*60)
print("\nActivation Function Comparison:")
print("-" * 40)
print(f"{'Activation':<15} {'Train Acc':<15} {'Test Acc':<15}")
print("-" * 40)
for activation in activations:
    train_acc = results[activation]['train_acc']
    test_acc = results[activation]['test_acc']
    print(f"{activation.capitalize():<15} {train_acc:.2f}%{'':<10} {test_acc:.2f}%")
print("-" * 40)

best_activation = max(results.keys(), key=lambda x: results[x]['test_acc'])
print(f"\nBest performing activation: {best_activation.upper()}")
print(f"Best test accuracy: {results[best_activation]['test_acc']:.2f}%")

print("\nKey Observations:")
print("- Sigmoid: Smooth gradients, outputs bounded [0, 1], good for binary classification")
print("- ReLU: Fast computation, helps with vanishing gradient, unbounded output")
print("- Tanh: Zero-centered, outputs bounded [-1, 1], often better than sigmoid")